In [1]:
import heapq

def print_state(build):
    for i in range(len(build)):
        stack = build[i].copy()
        temp = []
        while stack:
            temp.append(stack.pop())
        temp.reverse()
        print("Stack", i, ": ", end=" ")
        for block in temp:
            print(block, end=" ")
        print()

def input_state(s):
    print("Enter the details of the", s)
    n = int(input("Enter the number of stacks: "))
    build = [[] for _ in range(n)]
    for i in range(n):
        temp = int(input("Enter the number of blocks in stack "+ str(i) + ": "))
        if temp > 0:
            print("Enter the blocks :", end=" ")
        for j in range(temp):
            block = input().upper()
            build[i].append(block)
    return build

def copy(temp):
    result = temp.copy()
    return result

def serialize(state):
    result = ""
    for stack in state:
        for block in stack:
            result += block
        result += "|"
    return result

def clean_state(state):
    cleaned = []
    for stack in state:
        if len(stack) != 0:
            cleaned.append(stack)
    return cleaned

def heuristic_1(temp, goal):
    score = 0
    i = 0
    j = 0
    while i < len(temp) and j < len(goal):
        temp1 = copy(temp[i])
        temp2 = copy(goal[j])
        hi = 0
        hj = 0
        while hi < len(temp1) and hj < len(temp2):
            if temp1[hi] == temp2[hj]:
                score += 1
            else:
                score -= 1
            hi += 1
            hj += 1
        while hi < len(temp1):
            score -= 1
            hi += 1
        while hj < len(temp2):
            score -= 1
            hj += 1
        i += 1
        j += 1
    while i < len(temp):
        score -= len(temp[i])
        i += 1
    while j < len(goal):
        score -= len(goal[j])
        j += 1
    return score

def heuristic_2(temp, goal):
    score = 0
    i = 0
    j = 0
    while i < len(temp) and j < len(goal):
        temp1 = copy(temp[i])
        temp2 = copy(goal[j])
        hi = 0
        hj = 0
        while hi < len(temp1) and hj < len(temp2):
            n = hi + 1
            if temp1[hi] == temp2[hj]:
                score += n
            else:
                score -= n
            hi += 1
            hj += 1
        while hi < len(temp1):
            n = hi + 1
            score -= n
            hi += 1
        while hj < len(temp2):
            n = hj + 1
            score -= n
            hj += 1
        i += 1
        j += 1
    while i < len(temp):
        temp1 = copy(temp[i])
        for h in range(len(temp1)):
            score -= h + 1
        i += 1
    while j < len(goal):
        temp2 = copy(goal[j])
        for h in range(len(temp2)):
            score -= h + 1
        j += 1
    return score

def goal_test(temp, goal):
    if len(temp) != len(goal):
        return False
    for i in range(len(temp)):
        t1 = copy(temp[i])
        g1 = copy(goal[i])

        if t1 != g1:
            return False
    return True

def move_gen(start):
    neighbours = []
    n = len(start)
    for i in range(n):
        if len(start[i]) == 0:
            continue
        for j in range(n):
            if i == j:
                continue
            if len(start[j]) == 0:
                continue
            new_state = [stack.copy() for stack in start]
            block = new_state[i].pop()
            new_state[j].append(block)
            new_state = clean_state(new_state)
            neighbours.append(new_state)

        if len(start[i]) > 1:
            new_state_floor = [stack.copy() for stack in start]
            block = new_state_floor[i].pop()
            new_stack = [block]
            new_state_floor.append(new_stack)
            new_state_floor = clean_state(new_state_floor)
            neighbours.append(new_state_floor)
    return neighbours

def best_first_search(start, goal, heuristic_choice):
    priority_queue = []
    visited = set()
    parent_map = {}
    state_map = {}
    start_str = serialize(start)
    if heuristic_choice == 1:
        start_h = heuristic_1(start, goal)
    else:
        start_h = heuristic_2(start, goal)

    heapq.heappush(priority_queue,(-start_h, start_str, start))
    visited.add(start_str)
    parent_map[start_str] = None
    state_map[start_str] = start
    step = 0

    while priority_queue:
        negative_score, current_str, current_state = heapq.heappop(priority_queue)
        current_score = -negative_score
        print("\nExpanding Node",step,"(Heuristic Score:",current_score,")")
        print_state(current_state)
        step += 1
        if goal_test(current_state, goal):
            path = []
            backtrack_str = current_str
            while backtrack_str is not None:
                path.append(state_map[backtrack_str])
                backtrack_str = parent_map[backtrack_str]
            path.reverse()
            print("\nFINAL TRAVERSAL PATH (",len(path) - 1,"moves)")
            for i in range(len(path)):
                print("\nStep", i, ":")
                print_state(path[i])
                if i != len(path) - 1:
                    print("   |")
                    print("   V")
            return
        children = move_gen(current_state)
        for child in children:
            child_str = serialize(child)
            if child_str not in visited:
                visited.add(child_str)
                parent_map[child_str] = current_str
                state_map[child_str] = child
                if heuristic_choice == 1:
                    h = heuristic_1(child, goal)
                else:
                    h = heuristic_2(child, goal)
                heapq.heappush(priority_queue,(-h, child_str, child))

    print("\nFailed to reach the goal.")

def hill_climbing(start, goal, heuristic_choice):
    current_state = start
    path = [start]
    visited = set()
    visited.add(serialize(start))
    step = 0
    while True:
        if heuristic_choice == 1:
            current_score = heuristic_1(current_state,goal)
        else:
            current_score = heuristic_2(current_state,goal)

        print("\nExpanding Node",step,"(Heuristic Score:",current_score,")")
        print_state(current_state)
        step += 1
        if goal_test(current_state, goal):
            print("\nGOAL STATE REACHED!")
            print("\nFINAL TRAVERSAL PATH (",len(path) - 1,"moves)")

            for i in range(len(path)):
                print("\nStep", i, ":")
                print_state(path[i])
                if i != len(path) - 1:
                    print("   |")
                    print("   V")
            return

        children = move_gen(current_state)
        best_state = None
        best_score = current_score
        for child in children:
            child_str = serialize(child)
            if child_str in visited:
                continue
            if heuristic_choice == 1:
                h = heuristic_1(child, goal)
            else:
                h = heuristic_2(child, goal)
            if h > best_score:
                best_score = h
                best_state = child

        if best_state is None:
            print("\nHill Climbing stopped.")
            print("No better neighbour found` ")
            return

        best_str = serialize(best_state)
        if best_str in visited:
            print("\nHill Climbing stopped.")
            print("State already visited.")
            return

        current_state = best_state
        visited.add(best_str)
        path.append(current_state)

start = input_state("Start")
print()
goal = input_state("Goal")
print("\nInitial States")
print("Start State:")
print_state(start)
print("\nGoal State:")
print_state(goal)

Enter the details of the Start
Enter the blocks : Enter the blocks : 
Enter the details of the Goal
Enter the blocks : Enter the blocks : 
Initial States
Start State:
Stack 0 :  A B C 
Stack 1 :  D 
Stack 2 :  

Goal State:
Stack 0 :  A B 
Stack 1 :  D C 
Stack 2 :  


In [2]:
print("\nSelect Algorithm:")
print("1. Hill Climbing")
print("2. Best First Search")
algorithm_choice = int(input("Enter 1 or 2: "))

print("\nSelect Heuristic:")
print("1. Heuristic 1 ")
print("2. Heuristic 2 ")

heuristic_choice = int(input("Enter 1 or 2: "))

if algorithm_choice == 1:
    hill_climbing(start,goal,heuristic_choice)

elif algorithm_choice == 2:
    best_first_search(start,goal,heuristic_choice)

else:
    print("Invalid algorithm choice.")


Select Algorithm:
1. Hill Climbing
2. Best First Search

Select Heuristic:
1. Heuristic 1 
2. Heuristic 2 

Expanding Node 0 (Heuristic Score: 1 )
Stack 0 :  A B C 
Stack 1 :  D 
Stack 2 :  

Expanding Node 1 (Heuristic Score: 4 )
Stack 0 :  A B 
Stack 1 :  D C 

Expanding Node 2 (Heuristic Score: 1 )
Stack 0 :  A B C 
Stack 1 :  D 

Expanding Node 3 (Heuristic Score: 1 )
Stack 0 :  A B 
Stack 1 :  D 
Stack 2 :  C 

Expanding Node 4 (Heuristic Score: 1 )
Stack 0 :  A 
Stack 1 :  D C B 

Expanding Node 5 (Heuristic Score: 1 )
Stack 0 :  A 
Stack 1 :  D C 
Stack 2 :  B 

Expanding Node 6 (Heuristic Score: 0 )
Stack 0 :  A B 
Stack 1 :  C D 

Expanding Node 7 (Heuristic Score: -1 )
Stack 0 :  A B D 
Stack 1 :  C 

Expanding Node 8 (Heuristic Score: -1 )
Stack 0 :  A B 
Stack 1 :  C 
Stack 2 :  D 

Expanding Node 9 (Heuristic Score: -1 )
Stack 0 :  A C 
Stack 1 :  D 
Stack 2 :  B 

Expanding Node 10 (Heuristic Score: 0 )
Stack 0 :  A C 
Stack 1 :  D B 

Expanding Node 11 (Heuristic Score: